In [ ]:
# Ch07: 에이전트 기반 연구 파이프라인
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                          'pandapower', 'pypsa'])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(42)

try:
    sys.path.insert(0, '..')
    from utils.style import set_style, COLORS
    set_style()
except:
    pass

## 1. 경제 급전 (Economic Dispatch)

In [ ]:
from scipy.optimize import minimize

cost_coeff = np.array([
    [0.0430, 20.0, 0],
    [0.0250, 15.0, 0],
    [0.0100, 30.0, 0],
    [0.0120, 35.0, 0],
    [0.0080, 10.0, 0],
])
P_min = np.array([0, 0, 0, 0, 0])
P_max = np.array([332, 140, 100, 100, 100])
P_demand = 259.0

def total_cost(P):
    return sum(cost_coeff[i,0]*P[i]**2 + cost_coeff[i,1]*P[i] + cost_coeff[i,2] for i in range(5))

constraints = {'type': 'eq', 'fun': lambda P: sum(P) - P_demand}
bounds = list(zip(P_min, P_max))

result = minimize(total_cost, x0=[50]*5, method='SLSQP', bounds=bounds, constraints=constraints)
print(f"최적 발전량: {result.x.round(1)} MW")
print(f"최소 비용: {result.fun:.1f} $/h")

# 시각화
fig, ax = plt.subplots(figsize=(8, 5))
gen_names = ['G1\n(Slack)', 'G2', 'G3', 'G6', 'G8']
colors = ['#1B7A8A' if p > 0 else '#cccccc' for p in result.x]
ax.bar(gen_names, result.x, color=colors)
ax.set_ylabel('출력 (MW)')
ax.set_title(f'경제급전 결과 — 총 비용: {result.fun:.1f} $/h')
for i, v in enumerate(result.x):
    if v > 0:
        ax.text(i, v + 2, f'{v:.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 2. pandapower Monte Carlo 시뮬레이션

In [ ]:
import pandapower as pp
import pandapower.networks as pn

n_scenarios = 100
results = []

for i in range(n_scenarios):
    net = pn.case14()
    load_scale = 1.0 + np.random.uniform(-0.2, 0.2, len(net.load))
    net.load['p_mw'] *= load_scale
    net.load['q_mvar'] *= load_scale
    try:
        pp.runpp(net, verbose=False)
        results.append({
            'scenario': i,
            'v_min': net.res_bus['vm_pu'].min(),
            'v_max': net.res_bus['vm_pu'].max(),
            'total_loss': net.res_line['pl_mw'].sum(),
            'max_loading': net.res_line['loading_percent'].max(),
        })
    except:
        results.append({'scenario': i, 'converged': False})

df = pd.DataFrame(results).dropna()
print(f"수렴 시나리오: {len(df)}/{n_scenarios}")
print(f"전압 범위: {df['v_min'].min():.4f} ~ {df['v_max'].max():.4f} p.u.")
print(f"최대 선로 부하율: {df['max_loading'].max():.1f}%")

## 3. PyPSA 24시간 OPF

In [ ]:
import pypsa

n = pypsa.Network()
n.set_snapshots(range(24))

n.add("Bus", "bus_slack", v_nom=345)
n.add("Bus", "bus_load", v_nom=345)
n.add("Bus", "bus_wind", v_nom=345)

n.add("Generator", "gen_coal", bus="bus_slack", p_nom=500, marginal_cost=30, committable=False)

wind_cf = 0.3 + 0.2 * np.sin(np.linspace(0, 2*np.pi, 24))
n.add("Generator", "gen_wind", bus="bus_wind", p_nom=200, marginal_cost=0, p_max_pu=wind_cf, carrier="wind")

load_pattern = 300 + 100 * np.sin(np.linspace(0, 2*np.pi, 24))
n.add("Load", "load_1", bus="bus_load", p_set=load_pattern)

n.add("Line", "line_1", bus0="bus_slack", bus1="bus_load", s_nom=400, x=0.1, r=0.01)
n.add("Line", "line_2", bus0="bus_wind", bus1="bus_load", s_nom=300, x=0.15, r=0.02)

n.optimize(solver_name="highs")
print(f"총 비용: {n.objective:.0f} $")

# 발전 스택 시각화
fig, ax = plt.subplots(figsize=(10, 5))
coal = n.generators_t.p['gen_coal']
wind = n.generators_t.p['gen_wind']
ax.fill_between(range(24), 0, coal, alpha=0.7, label='석탄', color='#D4984A')
ax.fill_between(range(24), coal, coal + wind, alpha=0.7, label='풍력', color='#1B7A8A')
ax.plot(range(24), load_pattern, 'k-', lw=2, label='부하')
ax.set_xlabel('시간')
ax.set_ylabel('출력 (MW)')
ax.set_title('24시간 발전 스택')
ax.legend()
plt.tight_layout()
plt.show()

## 4. N-1 Contingency 분석

In [ ]:
def run_n1_analysis(net_func, v_limits=(0.95, 1.05), loading_limit=100):
    """N-1 contingency 분석 자동 수행"""
    results = []
    net = net_func()
    pp.runpp(net, verbose=False)
    
    for idx in net.line.index:
        net = net_func()
        net.line.at[idx, 'in_service'] = False
        try:
            pp.runpp(net, verbose=False)
            v_min = net.res_bus['vm_pu'].min()
            v_max = net.res_bus['vm_pu'].max()
            max_load = net.res_line['loading_percent'].max()
            results.append({
                'type': 'line', 'index': idx,
                'converged': True,
                'v_min': round(v_min, 4), 'v_max': round(v_max, 4),
                'max_loading': round(max_load, 2),
                # TODO: v_violation과 loading_violation 판정을 추가하세요
                'v_violation': None,  # <-- 구현하세요
                'loading_violation': None,  # <-- 구현하세요
            })
        except:
            results.append({'type': 'line', 'index': idx, 'converged': False,
                           'v_violation': True, 'loading_violation': True})
    
    return pd.DataFrame(results)

df_n1 = run_n1_analysis(pn.case14)
print(f"총 contingency: {len(df_n1)}")
print(df_n1[['index', 'v_min', 'v_max', 'max_loading']].to_string())

## 5. 민감도 분석 시나리오 매트릭스

In [ ]:
import itertools
from scipy.stats import qmc

params = {
    'load_scale': np.linspace(0.8, 1.2, 5),
    'wind_penetration': [0.1, 0.2, 0.3, 0.4],
    'gas_price': [20, 30, 40],
}

keys = list(params.keys())
combos = list(itertools.product(*params.values()))
scenarios = pd.DataFrame(combos, columns=keys)
print(f"Full factorial 시나리오 수: {len(scenarios)}")
print(scenarios.head(10))

# LHS 데모
sampler = qmc.LatinHypercube(d=len(keys), seed=42)
sample = sampler.random(n=50)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
random_pts = np.random.rand(50, 2)
axes[0].scatter(random_pts[:, 0], random_pts[:, 1], s=20, color='#C75C3A')
axes[0].set_title('단순 랜덤 샘플링')
axes[0].set_xlabel('파라미터 1')
axes[0].set_ylabel('파라미터 2')

axes[1].scatter(sample[:, 0], sample[:, 1], s=20, color='#1B7A8A')
axes[1].set_title('Latin Hypercube Sampling')
axes[1].set_xlabel('파라미터 1')
axes[1].set_ylabel('파라미터 2')

plt.tight_layout()
plt.show()